# Cache-aware compaction and continuation handoff

Compaction does not delete history. It appends a checkpoint to the ledger and changes only the model-facing projection. The current harness has a primary configurable trigger (80,000 estimated prompt tokens by default) and ADK event compaction as a later safety net (96,000 by default). It retains a recent raw tail and creates a structured continuation snapshot.

In [ ]:
import hashlib
import json


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


policy = {
    "primary_trigger_tokens": 80_000,
    "adk_backstop_tokens": 96_000,
    "retain_recent_events": 12,
}


def route(estimated_tokens):
    if estimated_tokens >= policy["adk_backstop_tokens"]:
        return "compact-now/backstop"
    if estimated_tokens >= policy["primary_trigger_tokens"]:
        return "compact-now/primary"
    return "continue"


assert route(79_999) == "continue"
assert route(80_000) == "compact-now/primary"
print({value: route(value) for value in (79_999, 80_000, 96_000)})

## The handoff is more than a summary

The deterministic contract carries goal and acceptance criteria, constraints/non-goals, done/in-progress/blocked work, decisions, exact code state, validation, next action, file sets, artifacts, and event boundaries. An LLM may improve semantic compression, but it cannot remove these fields or convert unknown work into completed work. Repeated compaction summarizes only the interval since the previous boundary and chains the previous snapshot hash.

In [ ]:
handoff = {
    "schema": "compaction.handoff@1",
    "goal": "Fix parser timeout without changing public syntax",
    "acceptance_criteria": ["parser tests pass", "public syntax unchanged"],
    "constraints": ["offline verification"],
    "progress": {
        "done": ["reproduced timeout"],
        "in_progress": ["bound recursive branch"],
        "blocked": [],
    },
    "decisions": [
        {"decision": "guard recursion in shared parser", "why": "all callers route through it"}
    ],
    "code_state": {
        "base_revision": "abc123",
        "files_read": ["parser.py", "tests/test_parser.py"],
        "files_modified": ["parser.py"],
    },
    "validation": {"last": "timeout", "command": "pytest -q tests/test_parser.py"},
    "next_action": "rerun the focused parser test with a 30 second bound",
    "open_execution": [{"kind": "test.run", "status": "timeout", "effect": "unknown"}],
    "boundaries": {"last_summarized_event_id": "evt-88", "first_retained_event_id": "evt-89"},
    "previous_summary_hash": "7be9...",
}
handoff_bytes = canonical(handoff).encode()
handoff_hash = hashlib.sha256(handoff_bytes).hexdigest()
assert hashlib.sha256(canonical(handoff).encode()).hexdigest() == handoff_hash
print(handoff["progress"], handoff["next_action"])
print(handoff_hash)

## Cache behavior: preserve what can actually remain stable

Provider caches are prefix caches. Normal turns append after the existing prompt and can reuse a large byte-identical prefix. Compaction replaces old dynamic history with a snapshot, so it necessarily starts a new dynamic cache epoch. The correct optimization is to keep P0 (system prompt, tool schema, stable project rules) byte-identical, serialize the new snapshot deterministically, and then append new turns. It is incorrect to claim that compaction preserves the entire pre-compaction cache.

In [ ]:
stable_prefix = canonical(
    {"P0": {"system": "coding-worker@7", "tools": ["python"], "project_rules_hash": "rules-v3"}}
)
old_history = canonical({"P1": {"epoch": "before", "events": list(range(1, 89))}})
new_history = canonical(
    {"P1": {"epoch": "after", "handoff_hash": handoff_hash, "retained": [89, 90, 91]}}
)
query = canonical({"P3": {"user": "continue"}})

before = stable_prefix + "\n" + old_history + "\n" + query
after = stable_prefix + "\n" + new_history + "\n" + query
repeat = stable_prefix + "\n" + new_history + "\n" + query

assert before.startswith(stable_prefix) and after.startswith(stable_prefix)
assert hashlib.sha256(after.encode()).digest() == hashlib.sha256(repeat.encode()).digest()
assert before != after
print(
    {
        "stable_prefix_bytes": len(stable_prefix.encode()),
        "new_epoch_byte_stable": after == repeat,
        "old_dynamic_cache_reusable": False,
    }
)

## Tunable strategies

Pi walks backward to a provider-valid cut, retains a recent token budget, incrementally updates its structured summary, and deterministically appends read/modified file lists. Codex selects among token-budget, provider-remote, and local model compaction; its local path retains selected recent user messages and a continuation summary. Both accept a cache discontinuity at compaction. The trace-native option can recompute most handoff fields from ledger views and reserve model summarization for semantic compression.

In [ ]:
strategies = [
    {
        "name": "deterministic_handoff",
        "status": "current fallback",
        "tail": "recent event count",
        "summary": "ledger-derived schema",
        "best_for": "replay and testability",
    },
    {
        "name": "pi_incremental",
        "status": "candidate",
        "tail": "recent token budget, valid call/result boundary",
        "summary": "previous summary + new interval + deterministic files",
        "best_for": "rich long conversations",
    },
    {
        "name": "codex_local_or_remote",
        "status": "provider-dependent candidate",
        "tail": "bounded selected messages",
        "summary": "local continuation summary or provider compaction item",
        "best_for": "provider-native optimization",
    },
    {
        "name": "trace_view_recompute",
        "status": "gated candidate",
        "tail": "recent raw events",
        "summary": "versioned progress/open/time/memory views + optional semantic capsule",
        "best_for": "auditable durable agents",
    },
]
for strategy in strategies:
    print(strategy["name"], "->", strategy["best_for"])
assert {item["name"] for item in strategies} == {
    "deterministic_handoff",
    "pi_incremental",
    "codex_local_or_remote",
    "trace_view_recompute",
}

## What to tune and score

Treat strategy, trigger, retained-tail budget, and semantic-summary budget as ablation variables. Compare pass rate, cost per passed task, uncached input tokens, cache-read ratio, prefix versions, tool calls, wall time, resume correctness, and loss of failure/open-work evidence. Do not switch the default based on token count alone.